# Sales Performance Investigation Using MySQL

## Business Scenario

Management has raised concerns about sales performance and wants to understand which salespeople and regions are underperforming, what patterns may be contributing to the weaker results, and what additional information is needed before making business decisions.

In this project, I use **MySQL** and SQL analysis to investigate sales performance across salespeople and regions.

### Questions investigated

- What are the overall sales, orders, returns, and average sales per order?
- Which salesperson generates the most and least sales?
- Does each salesperson generate a similar value per order?
- Which salesperson has the highest returns and discount levels?
- Which region performs better?
- Does the regional performance gap remain after accounting for the number of salespeople?
- Are regional differences driven by order volume, order value, or both?
- What additional data is needed to determine the underlying cause of weak performance?

**Tools:** MySQL, MySQL Workbench, SQL, GitHub/Jupyter Notebook


## `sales` table

The project began with a `sales` table created in MySQL. The table used for the analysis contains:

| Column | Definition |
|---|---|
| `order_id` | Unique order identifier |
| `salesperson` | Name of the salesperson |
| `region` | Region assigned to the salesperson |
| `sales` | Sales value |
| `orders` | Number of orders |
| `returns` | Number of returned orders |
| `discount` | Discount associated with the sale |

The CSV included with this repository is an export of the same table used in the MySQL analysis.


## 1. Overall Sales Performance

Before investigating the cause of weak performance, establish the overall sales position.

In [1]:
SELECT
    SUM(sales) AS total_sales,
    SUM(orders) AS total_orders,
    SUM(returns) AS total_returns,
    ROUND(SUM(sales) / SUM(orders), 2) AS avg_sales_per_order
FROM sales;

MySQL result:

|   total_sales |   total_orders |   total_returns |   avg_sales_per_order |
|--------------:|---------------:|----------------:|----------------------:|
|        280000 |             57 |              10 |               4912.28 |


### Observation

The dataset contains **KSh 280,000 in sales**, **57 orders**, and **9 returns**. Average sales per order are approximately **KSh 4,912.28**. These figures provide the baseline for the salesperson and regional comparisons that follow.


## 2. Sales Performance by Salesperson

Which salesperson contributes the most and least sales?

In [1]:
SELECT
    salesperson,
    SUM(sales) AS total_sales,
    SUM(orders) AS total_orders
FROM sales
GROUP BY salesperson
ORDER BY total_sales DESC;

MySQL result:

| salesperson   |   total_sales |   total_orders |
|:--------------|--------------:|---------------:|
| Carol         |        117000 |             23 |
| Alice         |         98000 |             19 |
| Brian         |         65000 |             15 |


### Observation

**Carol** generates the highest sales and orders, while **Brian** has the lowest sales and orders. Brian is therefore the main salesperson requiring further investigation.


## 3. Average Sales per Order by Salesperson

Are lower sales explained only by fewer orders, or is each order also generating less revenue?

In [1]:
SELECT
    salesperson,
    total_sales,
    total_orders,
    ROUND(total_sales / total_orders, 2) AS avg_sales_per_order
FROM (
    SELECT
        salesperson,
        SUM(sales) AS total_sales,
        SUM(orders) AS total_orders
    FROM sales
    GROUP BY salesperson
) AS salesperson_sales
ORDER BY total_sales DESC;

MySQL result:

| salesperson   |   total_sales |   total_orders |   avg_sales_per_order |
|:--------------|--------------:|---------------:|----------------------:|
| Carol         |        117000 |             23 |               5086.96 |
| Alice         |         98000 |             19 |               5157.89 |
| Brian         |         65000 |             15 |               4333.33 |


### Observation

Carol has the highest total sales, but **Alice has the highest average sales per order**. Brian has the lowest average sales per order. This suggests Brian's weaker performance is not only a volume issue; his orders also generate less revenue on average.


## 4. Returns and Discounts by Salesperson

Investigate whether weaker performance is accompanied by unusually high returns or discounting.

In [1]:
SELECT
    salesperson,
    SUM(sales) AS total_sales,
    SUM(orders) AS total_orders,
    SUM(returns) AS total_returns,
    AVG(discount) AS avg_discount
FROM sales
GROUP BY salesperson
ORDER BY total_sales DESC;

MySQL result:

| salesperson   |   total_sales |   total_orders |   total_returns |   avg_discount |
|:--------------|--------------:|---------------:|----------------:|---------------:|
| Carol         |        117000 |             23 |               2 |            5.5 |
| Alice         |         98000 |             19 |               3 |            6.5 |
| Brian         |         65000 |             15 |               5 |           11   |


### Observation

Brian has the highest average discount at **11.0%**, compared with **6.6%** for Alice and **5.5%** for Carol. This is a warning signal worth investigating, but the dataset does not prove that higher discounting causes Brian's weaker performance.


## 5. Return Rate by Salesperson

A high number of returns may reduce effective sales performance, so calculate returns as a percentage of orders.

In [1]:
SELECT
    salesperson,
    SUM(returns) AS total_returns,
    SUM(orders) AS total_orders,
    ROUND(SUM(returns) / SUM(orders) * 100, 2) AS return_rate
FROM sales
GROUP BY salesperson
ORDER BY return_rate DESC;

MySQL result:

| salesperson   |   total_returns |   total_orders |   return_rate |
|:--------------|----------------:|---------------:|--------------:|
| Brian         |               5 |             15 |         33.33 |
| Alice         |               3 |             19 |         15.79 |
| Carol         |               2 |             23 |          8.7  |


### Observation

Brian has a **33.33% return rate**, compared with **15.79%** for Alice and **8.70%** for Carol. The difference is large enough to warrant investigation. However, return rate alone does not establish the cause of weaker sales performance.


## 6. Regional Performance

Compare sales, orders, returns, discounting, and average sales per order by region.

In [1]:
SELECT
    region,
    total_sales,
    total_orders,
    total_returns,
    avg_discount,
    ROUND(total_sales / total_orders, 2) AS avg_sales_per_order
FROM (
    SELECT
        region,
        SUM(sales) AS total_sales,
        SUM(orders) AS total_orders,
        SUM(returns) AS total_returns,
        AVG(discount) AS avg_discount
    FROM sales
    GROUP BY region
) AS regional_sales;

MySQL result:

| region   |   total_sales |   total_orders |   total_returns |   avg_discount |   avg_sales_per_order |
|:---------|--------------:|---------------:|----------------:|---------------:|----------------------:|
| Eastern  |         65000 |             15 |               5 |             11 |               4333.33 |
| Western  |        215000 |             42 |               5 |              6 |               5119.05 |


### Observation

Western generates **KSh 215,000** compared with **KSh 65,000** in Eastern. Western also has the higher average sales per order, while Eastern has the higher average discount and return rate.


## 7. Normalize Regional Performance by Salesperson Count

Western has two salespeople while Eastern has one. Comparing raw regional totals alone could therefore be misleading.

In [1]:
SELECT
    region,
    total_sales,
    number_of_salesperson,
    ROUND(total_sales / number_of_salesperson, 2) AS sales_per_person
FROM (
    SELECT
        region,
        SUM(sales) AS total_sales,
        COUNT(DISTINCT salesperson) AS number_of_salesperson
    FROM sales
    GROUP BY region
) AS sales_per_salesperson_per_region;

MySQL result:

| region   |   total_sales |   number_of_salesperson |   sales_per_person |
|:---------|--------------:|------------------------:|-------------------:|
| Eastern  |         65000 |                       1 |              65000 |
| Western  |        215000 |                       2 |             107500 |


### Observation

Eastern generates **KSh 65,000 per salesperson**, compared with **KSh 107,500** in Western. Eastern therefore remains weaker even after accounting for the different number of salespeople.


## 8. Orders per Salesperson by Region

Determine whether Eastern's weaker sales are also associated with lower order productivity.

In [1]:
SELECT
    region,
    total_orders,
    number_of_salesperson,
    ROUND(total_orders / number_of_salesperson, 2) AS orders_per_person
FROM (
    SELECT
        region,
        SUM(orders) AS total_orders,
        COUNT(DISTINCT salesperson) AS number_of_salesperson
    FROM sales
    GROUP BY region
) AS sales_per_salesperson_per_region;

MySQL result:

| region   |   total_orders |   number_of_salesperson |   orders_per_person |
|:---------|---------------:|------------------------:|--------------------:|
| Eastern  |             15 |                       1 |                  15 |
| Western  |             42 |                       2 |                  21 |


### Observation

Eastern generates **15 orders per salesperson**, compared with **21** in Western. This indicates lower order productivity in Eastern.


## 9. Average Sales per Order by Region

Separate the regional sales difference into order volume and order value.

In [1]:
SELECT
    region,
    total_sales,
    total_orders,
    ROUND(total_sales / total_orders, 2) AS avg_sales_per_order
FROM (
    SELECT
        region,
        SUM(sales) AS total_sales,
        SUM(orders) AS total_orders
    FROM sales
    GROUP BY region
) AS regional_sales_per_order;

MySQL result:

| region   |   total_sales |   total_orders |   avg_sales_per_order |
|:---------|--------------:|---------------:|----------------------:|
| Eastern  |         65000 |             15 |               4333.33 |
| Western  |        215000 |             42 |               5119.05 |


### Observation

Eastern's weaker sales are associated with **both lower order volume and lower average revenue per order**. Eastern averages approximately **KSh 4,333.33 per order**, compared with **KSh 5,119.05** in Western.


## Key Findings

1. **Brian is the primary salesperson requiring investigation.** He has the lowest sales, fewest orders, lowest average sales per order, highest average discount, and highest return rate.

2. **Eastern is the weaker region.** Even after adjusting for salesperson count, Eastern generates KSh 65,000 per salesperson compared with KSh 107,500 in Western.

3. **Eastern's weaker performance is not explained by salesperson count alone.** Eastern has fewer orders per salesperson (15 vs. 21) and lower average sales per order (~KSh 4,333 vs. ~KSh 5,119).

4. **High discounting and returns are warning signals.** Brian's 11% average discount and 33.33% return rate deserve further investigation, but the current data does not prove causation.


## Root-Cause Hypotheses

The current dataset identifies **where** performance is weak, but it does not establish **why** it is weak.

Potential factors to investigate include:

- Sales strategy and sales channels
- Product mix and product prices
- Customer segments
- Customer acquisition and sales opportunities
- Customer service and satisfaction
- Reasons for returned products
- Product handling and delivery quality
- Seasonal or time-based sales patterns

These are hypotheses, not confirmed causes.


## Recommendations

### Immediate

- Investigate Brian's unusually high return rate and discounting.
- Review sales activity and order-generation methods in Eastern.
- Compare products and customer types served in Eastern with those in Western.

### Further analysis

Collect or analyze additional data on **product category, customer segment, sales channel, return reason, customer satisfaction, and sales date/month**. This would allow management to distinguish salesperson effects from regional or market effects.

### Important limitation

Brian is the only salesperson in Eastern. Therefore, this dataset cannot cleanly separate the effect of the salesperson from the effect of the Eastern market. The findings should be treated as evidence for further investigation rather than proof of a single root cause.


## Conclusion

The SQL investigation identifies **Brian and the Eastern region** as the main areas requiring attention. Eastern underperforms Western not only in total sales but also in sales per salesperson, orders per salesperson, and average sales per order. Brian also shows unusually high discounting and returns.

However, the available data is insufficient to establish the underlying cause. Additional product, customer, sales-channel, return-reason, customer-service, and time-based data would be required for a stronger root-cause analysis.

**Project tools:** MySQL, MySQL Workbench, SQL, GitHub
